In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../docs/df_features.csv')
print(df.shape)
print(df['hypertension_grade'].value_counts())

(8777, 31)
hypertension_grade
0    3512
1    2181
2    1558
3    1526
Name: count, dtype: int64


In [4]:
# Target variable
y = df['hypertension_grade']

# Features - lifestyle and demographic only
X = df.drop(columns=[
    'id', 'country_name', 'age_group', 'sex',
    'sbp_avg', 'dbp_avg', 'hypertension_grade',
    'survey_weight',
])

print("Features:", X.columns.tolist())
print("Shape:", X.shape)

Features: ['country', 'residence', 'age', 'height_cm', 'weight_kg', 'waist_cm', 'hip_cm', 'ever_tobacco', 'current_tobacco', 'ever_alcohol', 'recent_alcohol', 'avg_drinks_daily', 'vigorous_work', 'days_vigorous_work', 'moderate_work', 'walks_or_cycles', 'vigorous_leisure', 'moderate_leisure', 'fruit_servings', 'veg_servings', 'bmi', 'sex_encoded', 'age_sex_interaction']
Shape: (8777, 23)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nClass distribution in training set:")
print(y_train.value_counts())

Training set: (7021, 23)
Test set: (1756, 23)

Class distribution in training set:
hypertension_grade
0    2809
1    1745
2    1246
3    1221
Name: count, dtype: int64


In [6]:
print(X_train['residence'].unique())

[2 1]


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(
        n_estimators=100, random_state=42, eval_metric='mlogloss')
}

# Train and evaluate each
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print('='*50)
    
    # Use scaled data for logistic regression, unscaled for tree models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    results[name] = y_pred
    print(classification_report(y_test, y_pred,
          target_names=['Normal', 'Grade 1', 'Grade 2', 'Grade 3']))


Model: Logistic Regression
              precision    recall  f1-score   support

      Normal       0.53      0.54      0.53       703
     Grade 1       0.24      0.14      0.18       436
     Grade 2       0.19      0.16      0.17       312
     Grade 3       0.23      0.38      0.28       305

    accuracy                           0.35      1756
   macro avg       0.30      0.31      0.29      1756
weighted avg       0.34      0.35      0.34      1756


Model: Random Forest
              precision    recall  f1-score   support

      Normal       0.47      0.78      0.59       703
     Grade 1       0.37      0.25      0.29       436
     Grade 2       0.36      0.13      0.19       312
     Grade 3       0.27      0.17      0.21       305

    accuracy                           0.43      1756
   macro avg       0.37      0.33      0.32      1756
weighted avg       0.39      0.43      0.38      1756


Model: XGBoost
              precision    recall  f1-score   support

      Nor

In [10]:
df['hypertensive'] = (df['hypertension_grade'] > 0).astype(int)
print(df['hypertensive'].value_counts())

hypertensive
1    5265
0    3512
Name: count, dtype: int64


In [11]:
# New target
y = df['hypertensive']

# Same features as before
X = df.drop(columns=[
    'id', 'country_name', 'age_group', 'sex',
    'sbp_avg', 'dbp_avg', 'hypertension_grade',
    'survey_weight', 'hypertensive'
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train all three models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(
        n_estimators=100, random_state=42, eval_metric='logloss')
}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print('='*50)
    
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred,
          target_names=['Normal', 'Hypertensive']))


Model: Logistic Regression
              precision    recall  f1-score   support

      Normal       0.52      0.63      0.57       703
Hypertensive       0.71      0.62      0.66      1053

    accuracy                           0.62      1756
   macro avg       0.62      0.62      0.62      1756
weighted avg       0.64      0.62      0.63      1756


Model: Random Forest
              precision    recall  f1-score   support

      Normal       0.62      0.37      0.47       703
Hypertensive       0.67      0.85      0.75      1053

    accuracy                           0.66      1756
   macro avg       0.64      0.61      0.61      1756
weighted avg       0.65      0.66      0.64      1756


Model: XGBoost
              precision    recall  f1-score   support

      Normal       0.57      0.44      0.50       703
Hypertensive       0.68      0.78      0.72      1053

    accuracy                           0.64      1756
   macro avg       0.62      0.61      0.61      1756
weighted

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 300}
Best score: 0.6434851924337985


In [13]:
# Train final tuned model
final_model = RandomForestClassifier(
    max_depth=None,
    min_samples_split=10,
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
)

final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)

print(classification_report(y_test, y_pred_final,
      target_names=['Normal', 'Hypertensive']))

              precision    recall  f1-score   support

      Normal       0.60      0.45      0.52       703
Hypertensive       0.69      0.80      0.74      1053

    accuracy                           0.66      1756
   macro avg       0.64      0.63      0.63      1756
weighted avg       0.65      0.66      0.65      1756



In [14]:
import joblib

# Save model and scaler
joblib.dump(final_model, '../models/hypertension_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')

print("Model saved successfully")

Model saved successfully
